# Mission 02: Multi-layered Prompt & Cache - 해답 노트북

이 노트북은 다섯 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [ ]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from langchain_core.messages import SystemMessage, HumanMessage

### [미션 1] 계층형 PromptManager 구현하기

아래 코드는 L1 ~ L4의 계층별 프롬프트 블록을 합쳐 시스템 프롬프트를 렌더링하고 동적 데이터를 치환하는 PromptManager 솔루션 코드입니다.

In [ ]:
# prompts 폴더 하위의 prompt_manager 모듈로부터 PromptManager 클래스 로드
import sys
import os

# 현재 폴더의 prompts 패키지 임포트를 가능하게 경로 추가
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

from prompts.prompt_manager import PromptManager

pm = PromptManager()
print("✅ 파일 기반 PromptManager 로드 및 객체 초기화 성공!")


### [미션 2] 프롬프트 렌더링 및 캐시 지정 시뮬레이션

대용량 더미 문서 데이터를 L4 영역에 설정하고, `PromptManager`를 통해 시스템 프롬프트를 구성해 봅니다.

In [ ]:
# 대규모 정적 API 레퍼런스 문서 모사 (캐싱 대상)
large_api_docs = """=== API Reference Manual ===\n""" + "\n".join(
    [f"Function_ID_{i}: Perform operation {i}. Parameters: arg{i}. Returns result." for i in range(500)]
)

pm.set_reference_context(large_api_docs)

state = {
    "user_permission": "READ_WRITE_EXECUTE",
    "active_project": "harness_agent_lab"
}

system_prompt = pm.build_system_prompt(state)

print(f"생성된 시스템 프롬프트 크기: {len(system_prompt)} 글자")
print("상단 300글자 요약:\n", system_prompt[:300])

### [미션 3] 에이전트 캐싱 지연시간 최적화 검증

캐싱이 걸렸을 때의 Latency 단축 효과를 확인하기 위해 시뮬레이션 테스트를 수행합니다.

In [ ]:
import time
llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)

print("🔄 1회차 호출 (Cold Start - 캐시 생성) 시작...")
t1 = time.time()
res1 = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="Function_ID_256번 API의 매개변수와 반환 스펙이 무엇인지 설명해줘.")
])
cold_latency = time.time() - t1
print(f"✅ 1회차 호출 완료! (소요 시간: {cold_latency:.2f}초)")
print("답변:", res1.content)

print("\n" + "-"*50 + "\n")

print("🔄 2회차 호출 (Warm Start - 캐시 히트) 시작...")
t2 = time.time()
res2 = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="Function_ID_128번 API의 매개변수와 반환 스펙은 뭐야?")
])
warm_latency = time.time() - t2
print(f"✅ 2회차 호출 완료! (소요 시간: {warm_latency:.2f}초)")
print("답변:", res2.content)

speedup = cold_latency / warm_latency if warm_latency > 0 else 1.0
print(f"\n🚀 캐싱 적용으로 속도가 약 {speedup:.1f}배 개선되었습니다.")